# 0. Environment Setup

In [11]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import torch
import torchvision
from torch.utils import data
from torchvision import transforms
import matplotlib.pyplot as plt

DATA_PATH = "./data"

# 1. Load Data from Fashion-MNIST Dataset

In [12]:
trans = transforms.ToTensor()
mnist_train = torchvision.datasets.FashionMNIST(root=DATA_PATH, train=True, download=True, transform=trans)
mnist_test = torchvision.datasets.FashionMNIST(root=DATA_PATH, train=False, download=True, transform=trans)
print(f"Number of training examples: {len(mnist_train)}")
print(f"Number of testing examples: {len(mnist_test)}")

Number of training examples: 60000
Number of testing examples: 10000


In [13]:
batch_size = 256

def get_dataloader_workers():
    return 0

train_iter = data.DataLoader(
    mnist_train,
    batch_size,
    shuffle=True,
    num_workers=get_dataloader_workers()
)

test_iter = data.DataLoader(
    mnist_test,
    batch_size,
    shuffle=True,
    num_workers=get_dataloader_workers()
)

In [14]:
class Accumulator:
    def __init__(self, n):
        self.data = [0.0] * n

    def add(self, *args):
        self.data = [a + float(b) for a, b in zip(self.data, args)]

    def reset(self):
        self.data = [0.0] * len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def accuracy(y_hat, y):
    if len(y_hat.shape) >1 and y_hat.shape[1] > 1:
        y_hat = y_hat.argmax(axis = 1)
    cmp = y_hat.type(y.dtype) == y
    return float(cmp.type(y.dtype).sum())

def evaluate_accuracy(net, data_iter):
    if isinstance(net, torch.nn.Module):
        net.eval()
    metric = Accumulator(2)  # 正确预测数、预测总数
    with torch.no_grad():
        for X, y in data_iter:
            metric.add(accuracy(net(X), y), y.numel())
    return metric[0] / metric[1]

In [15]:
from sklearn.base import ClassifierMixin, BaseEstimator

class mySoftmaxClassifier(ClassifierMixin, BaseEstimator):
    def __init__(self, lr, num_epochs=1000):
        self._w = None
        self._b = 0
        self._lr = lr
        self._num_epochs = num_epochs

    def fit(self, X, y):
        raise NotImplementedError("This method is not implemented yet.")
    
    def my_fit(self, data_iter, num_input, num_output):
        self._w = torch.normal(0, 0.01, size=(num_input, num_output), requires_grad=True)
        self._b = torch.zeros(num_output, requires_grad=True)
        metric = Accumulator(3) # Sum of training loss, Sum of training accuracy, Number of examples
        last_acc = 0
        for epoch in range(self._num_epochs):
            for X, y in data_iter:
                y_hat = self.net(X)
                l = self.cross_entropy(y_hat, y)
                l.sum().backward()
                with torch.no_grad():
                    self._w -= self._lr * self._w.grad / X.shape[0]
                    self._b -= self._lr * self._b.grad / X.shape[0]
                    self._w.grad.zero_()
                    self._b.grad.zero_()
                    metric.add(l.sum(), accuracy(y_hat, y), y.numel())
            cur_acc = metric[1] / metric[2]
            if abs(cur_acc - last_acc) < 1e-4:
                print(f"Early stopping at epoch {epoch+1} with training accuracy {cur_acc:.3f}")
                break
            last_acc = cur_acc
            if (epoch + 1) % 100 == 0:
                print(f"Epoch: {epoch+1}, loss: {metric[0] / metric[2]:.3f}, train acc: {cur_acc:.3f}")
    
    def predict(self, X):
        return self.net(X)
        
    def net(self, X):
        return self.softmax(
            X.reshape(-1, self._w.shape[0]) @ self._w + self._b
        )
    
    def softmax(self, X):
        X_max = X.max(dim=1, keepdim=True)[0]
        X = X - X_max
        X_exp = torch.exp(X)
        partition = X_exp.sum(dim=1, keepdim=True)
        return X_exp / partition
    
    def cross_entropy(self, y_hat, y):
        return -torch.log(
            y_hat[range(len(y_hat)), y]
            )
        
        
msc = mySoftmaxClassifier(0.01)
msc.my_fit(train_iter, 28*28, 10)
print(evaluate_accuracy(
    msc.net, test_iter
))

Epoch: 100, loss: 0.509, train acc: 0.833
Early stopping at epoch 155 with training accuracy 0.840
0.839
